In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [97]:
data = pd.read_csv("train.csv")

## Raw Data EDA

In [98]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       690088 non-null  int64  
 1   health_condition         690088 non-null  object 
 2   sleep_duration           614089 non-null  float64
 3   heart_rate               682255 non-null  float64
 4   bmi                      676190 non-null  float64
 5   calorie_expenditure      637235 non-null  float64
 6   step_count               676172 non-null  float64
 7   exercise_duration        683187 non-null  float64
 8   water_intake             646611 non-null  float64
 9   diet_type                683187 non-null  object 
 10  stress_level             607277 non-null  object 
 11  sleep_quality            631757 non-null  object 
 12  physical_activity_level  653467 non-null  object 
 13  smoking_alcohol          661506 non-null  object 
 14  gend

In [99]:
data.isnull().sum()

id                             0
health_condition               0
sleep_duration             75999
heart_rate                  7833
bmi                        13898
calorie_expenditure        52853
step_count                 13916
exercise_duration           6901
water_intake               43477
diet_type                   6901
stress_level               82811
sleep_quality              58331
physical_activity_level    36621
smoking_alcohol            28582
gender                     21373
dtype: int64

In [100]:
data.duplicated().sum()   # no duplicates

np.int64(0)

In [101]:
cat_cols = data.select_dtypes(include = ['object']).columns  # categorical cols
num_cols = data.select_dtypes(include = ['float64', 'int64']).columns  # numerical cols

## Preprocessing

In [102]:
# Handling missing values

#------missing indicators-------------
data['missing_sleep_duration'] = data['sleep_duration'].isnull().astype(int)
data['missing_heart_rate'] = data['heart_rate'].isnull().astype(int)
data['missing_bmi'] = data['bmi'].isnull().astype(int)
data['missing_calorie_expenditure'] = data['calorie_expenditure'].isnull().astype(int)
data['missing_step_count'] = data['step_count'].isnull().astype(int)
data['missing_exercise_duration'] = data['exercise_duration'].isnull().astype(int)
data['missing_water_intake'] = data['water_intake'].isnull().astype(int)
data['missing_diet_type'] = data['diet_type'].isnull().astype(int)
data['missing_stress_level'] = data['stress_level'].isnull().astype(int)
data['missing_sleep_quality'] = data['sleep_quality'].isnull().astype(int)
data['missing_physical_activity_level'] = data['physical_activity_level'].isnull().astype(int)
data['missing_smoking_alcohol'] = data['smoking_alcohol'].isnull().astype(int)
data['missing_gender'] = data['gender'].isnull().astype(int)

#------ fill missing values-----------
data['sleep_duration'] = data['sleep_duration'].fillna(data['sleep_duration'].median())
data['heart_rate'] = data['heart_rate'].fillna(data['heart_rate'].median())
data['bmi'] = data['bmi'].fillna(data['bmi'].median())
data['calorie_expenditure'] = data['calorie_expenditure'].fillna(data['calorie_expenditure'].median())
data['step_count'] = data['step_count'].fillna(data['step_count'].median())
data['exercise_duration'] = data['exercise_duration'].fillna(data['exercise_duration'].median())
data['water_intake'] = data['water_intake'].fillna(data['water_intake'].median())
data['diet_type'] = data['diet_type'].fillna(data['diet_type'].mode()[0])
data['stress_level'] = data['stress_level'].fillna(data['stress_level'].mode()[0])
data['sleep_quality'] = data['sleep_quality'].fillna(data['sleep_quality'].mode()[0])
data['physical_activity_level'] = data['physical_activity_level'].fillna(data['physical_activity_level'].mode()[0])
data['smoking_alcohol'] = data['smoking_alcohol'].fillna(data['smoking_alcohol'].mode()[0])
data['gender'] = data['gender'].fillna(data['gender'].mode()[0])


In [103]:
data.select_dtypes(include = ['object']).columns

Index(['health_condition', 'diet_type', 'stress_level', 'sleep_quality',
       'physical_activity_level', 'smoking_alcohol', 'gender'],
      dtype='object')

In [104]:
X = data.drop(columns = ['id','health_condition'], axis = 1)
y = data['health_condition']

In [105]:
# encoding
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

#-----Label Encoding--------
le = LabelEncoder()
y = le.fit_transform(y)

#-----manual mapping--------
stress_mapping = {
    'low' : 0,
    'medium' : 1,
    'high' : 2
}
X['stress_level'] = X['stress_level'].map(stress_mapping)

sleep_mapping = {
    'poor' : 0,
    'average' : 1,
    'good' : 2
}
X['sleep_quality'] = X['sleep_quality'].map(sleep_mapping)

physical_mapping = {
    'sedentary' : 0,
    'moderate' : 1,
    'active' : 2
}
X['physical_activity_level'] = X['physical_activity_level'].map(physical_mapping)

#-----OHE------------
ohe = OneHotEncoder(
    drop = 'first',
    handle_unknown = 'ignore',
    sparse_output = False
)
encoded = ohe.fit_transform(X[['diet_type', 'smoking_alcohol', 'gender']])
encoded_data = pd.DataFrame(encoded, columns = ohe.get_feature_names_out(), index = X.index)
X = pd.concat([X.drop(columns = ['diet_type', 'smoking_alcohol', 'gender']), encoded_data], axis = 1)

In [11]:
results = {}
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import log_loss, accuracy_score, recall_score, f1_score, precision_score, balanced_accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42, stratify=y
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

def train_n_eval(model, model_name):
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)
    best_params = model.best_params_

    evaluation = f"Balanced accuracy : {balanced_accuracy_score(y_test, y_pred)}\nlog_loss : {log_loss(y_test, y_prob)}\nTraining_acc : {accuracy_score(y_train, y_train_pred)}\nTesting_acc : {accuracy_score(y_test, y_pred)}\nPrecision : {precision_score(y_test, y_pred, average='weighted')}\nf1 score : {f1_score(y_test, y_pred, average='weighted')} \nRecall : {recall_score(y_test, y_pred, average='weighted')}"

    results[model_name] = {
        "Balanced accuracy" : balanced_accuracy_score(y_test, y_pred),
        "log_loss" : log_loss(y_test, y_prob),
        "Accuracy" : accuracy_score(y_test, y_pred),
        "Precision" : precision_score(y_test, y_pred, average='weighted'),
        "f1 score" : f1_score(y_test, y_pred, average='weighted'),
        "Recall" : recall_score(y_test, y_pred, average='weighted'),
        "Best_param" : best_params
    }
    print(best_params)
    print(evaluation)

## Logistic Regression

In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


lr_pipeline = Pipeline(
    [
    ("scaling", StandardScaler()),
    ("estimator",LogisticRegression(max_iter = 5000, random_state = 42))
    ]
)
params = {
    'estimator__C' : [0.001,0.01,0.1,1,10,50]
}

logistic_grid = GridSearchCV(
    lr_pipeline,
    params,
    cv = cv,
    scoring = 'balanced_accuracy',
    refit = True,
    n_jobs = -1,
    verbose=2
)

train_n_eval(logistic_grid, "Logistic Regression")

Fitting 5 folds for each of 6 candidates, totalling 30 fits
{'estimator__C': 10}
Balanced accuracy : 0.8353246846736813
log_loss : 0.17482933987744517
Training_acc : 0.9517472167800821
Testing_acc : 0.9519829355096742
Precision : 0.950454938350122
f1 score : 0.9504857862971393 
Recall : 0.9519829355096742


## Gaussian NB

In [13]:
from sklearn.naive_bayes import GaussianNB

gnb_pipeline = Pipeline(
    [
        ("estimator", GaussianNB())
    ]
)
params= {
        "estimator__var_smoothing" : [1e-11, 1e-10, 1e-9, 1e-8, 1e-7]
}

gaussian_grid = GridSearchCV(
    gnb_pipeline,
    params,
    cv = cv,
    refit = True,
    scoring = 'balanced_accuracy',
    n_jobs = -1,
    verbose = 2
)

train_n_eval(gaussian_grid, "Gaussian NB")

Fitting 5 folds for each of 5 candidates, totalling 25 fits
{'estimator__var_smoothing': 1e-11}
Balanced accuracy : 0.8089275471116312
log_loss : 0.3063591906048674
Training_acc : 0.9099168028811785
Testing_acc : 0.9107592075213596
Precision : 0.9175102787903743
f1 score : 0.9132227829529521 
Recall : 0.9107592075213596


## Decision Tree

In [14]:
from sklearn.tree import DecisionTreeClassifier

full_tree = DecisionTreeClassifier(random_state=42)
full_tree.fit(X_train, y_train)
path = full_tree.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas

def find_ccps(n_candidates, alphas):
    sample_list = []

    chunk_size = max(1, len(alphas)//n_candidates)

    for i in range(0, len(alphas), chunk_size):
        chunk = alphas[i:i+chunk_size]
        alpha = np.median(chunk)
        sample_list.append(alpha)

    return np.unique(sample_list)

ccp_list = find_ccps(23, ccp_alphas)

dtc_pipeline = Pipeline(
    [
        ("estimator", DecisionTreeClassifier(random_state = 42))
    ]
)

params = {
    "estimator__ccp_alpha" : ccp_list
}

dtc_grid = GridSearchCV(
    dtc_pipeline,
    params,
    cv=cv,
    refit = True,
    scoring = 'balanced_accuracy',
    n_jobs = -1,
    verbose = 2
)

train_n_eval(dtc_grid, "Decision Tree")

Fitting 5 folds for each of 22 candidates, totalling 110 fits
{'estimator__ccp_alpha': np.float64(3.512946789746271e-06)}
Balanced accuracy : 0.8841506509015854
log_loss : 0.5395322998085064
Training_acc : 0.9857795913951071
Testing_acc : 0.9609151296646224
Precision : 0.9602612856647998
f1 score : 0.9604955754412418 
Recall : 0.9609151296646224


## Random Forest

In [15]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline(
    [
        ("estimator", RandomForestClassifier(n_jobs=1, random_state=42))
    ]
)

params = {
    "estimator__n_estimators" : [100],
    "estimator__max_depth" : [None],
    "estimator__max_features" : ["sqrt"]
}

rf_grid = GridSearchCV(
    estimator = rf_pipeline,
    param_grid=params,
    cv=StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
        ),
    refit = True,
    scoring = 'balanced_accuracy',
    n_jobs = -1,
    verbose = 2
)

train_n_eval(rf_grid, "Random Forest")

Fitting 3 folds for each of 1 candidates, totalling 3 fits
{'estimator__max_depth': None, 'estimator__max_features': 'sqrt', 'estimator__n_estimators': 100}
Balanced accuracy : 0.8614415546460211
log_loss : 0.18706335007771652
Training_acc : 0.999978746671922
Testing_acc : 0.9660391138521464
Precision : 0.9658358033261766
f1 score : 0.9646578558112654 
Recall : 0.9660391138521464


## Extra Tree

In [16]:
from sklearn.ensemble import ExtraTreesClassifier

et_pipeline = Pipeline(
    [
        ("estimator", ExtraTreesClassifier(random_state=42))
    ]
)

params = {
    "estimator__n_estimators" : [100],
    "estimator__max_features" : ["sqrt"]
}

et_grid = GridSearchCV(
    et_pipeline,
    params,
    cv=cv,
    refit=True,
    n_jobs = -1,
    scoring="balanced_accuracy",
    verbose=2
)
train_n_eval(et_grid, "Extra Tree(n_e = 100)")

Fitting 5 folds for each of 1 candidates, totalling 5 fits
{'estimator__max_features': 'sqrt', 'estimator__n_estimators': 100}
Balanced accuracy : 0.8519550080673276
log_loss : 0.2143107445701801
Training_acc : 1.0
Testing_acc : 0.9634771217583844
Precision : 0.9631491599832048
f1 score : 0.9619041972337573 
Recall : 0.9634771217583844


## Gradient Boosting

In [17]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb_pipeline = Pipeline(
    [
        ("estimator", HistGradientBoostingClassifier(random_state=42))
    ]
)

params = {
    "estimator__learning_rate" : [0.001, 0.01, 0.1],
    "estimator__max_iter" : [100, 300],
    "estimator__max_depth" : [3,5]
}

gb_grid = GridSearchCV(
    gb_pipeline,
    params,
    cv=cv,
    refit = True,
    n_jobs = -1,
    scoring = "balanced_accuracy",
    verbose = 2
)

train_n_eval(gb_grid, "Gradient Boosting")

Fitting 5 folds for each of 12 candidates, totalling 60 fits
{'estimator__learning_rate': 0.1, 'estimator__max_depth': 3, 'estimator__max_iter': 300}
Balanced accuracy : 0.8752342871887523
log_loss : 0.09042438448176504
Training_acc : 0.968577920497096
Testing_acc : 0.967505593489526
Precision : 0.9670324226094622
f1 score : 0.9664666279247228 
Recall : 0.967505593489526


In [18]:

params = {
    "estimator__learning_rate" : [0.08, 0.01, 0.15],
    "estimator__max_iter" : [280, 300, 320],
    "estimator__max_depth" : [4,5,6]
}

gb_grid2 = GridSearchCV(
    gb_pipeline,
    params,
    cv=cv,
    refit = True,
    n_jobs = -1,
    scoring = "balanced_accuracy",
    verbose = 2
)

train_n_eval(gb_grid2, "Gradient Boosting v2")

Fitting 5 folds for each of 27 candidates, totalling 135 fits
{'estimator__learning_rate': 0.08, 'estimator__max_depth': 4, 'estimator__max_iter': 280}
Balanced accuracy : 0.876218238637367
log_loss : 0.0899408906803289
Training_acc : 0.9691652852003416
Testing_acc : 0.9672505535525904
Precision : 0.9667400480974305
f1 score : 0.9662360771207815 
Recall : 0.9672505535525904


## XG Boost

In [19]:
import xgboost as xgb

xgb_pipeline = Pipeline(
    [
        ("estimator", xgb.XGBClassifier(random_state = 42))
    ]
)

params = {
    "estimator__learning_rate" : [0.001, 0.01, 0.1],
    "estimator__n_estimators" : [100, 300],
    "estimator__max_depth" : [3,5]
    
}

xgb_grid = GridSearchCV(
    xgb_pipeline,
    params,
    cv=cv,
    refit = True,
    n_jobs = -1,
    scoring = "balanced_accuracy",
    verbose = 2
)

train_n_eval(xgb_grid, "XG Boost")

Fitting 5 folds for each of 12 candidates, totalling 60 fits
{'estimator__learning_rate': 0.1, 'estimator__max_depth': 5, 'estimator__n_estimators': 300}
Balanced accuracy : 0.8773174622567129
log_loss : 0.0893963656504052
Training_acc : 0.9709138544649378
Testing_acc : 0.9674766116785105
Precision : 0.9669513753675978
f1 score : 0.9664946237639945 
Recall : 0.9674766116785105


In [20]:
params = {
    "estimator__learning_rate" : [0.08, 0.1,0.15],
    "estimator__n_estimators" : [250,300,320],
    "estimator__max_depth" : [4, 5, 6]
    
}

xgb_grid2 = GridSearchCV(
    xgb_pipeline,
    params,
    cv=cv,
    refit = True,
    n_jobs = -1,
    scoring = "balanced_accuracy",
    verbose = 2
)

train_n_eval(xgb_grid2, "XG Boost v2")

Fitting 5 folds for each of 27 candidates, totalling 135 fits
{'estimator__learning_rate': 0.15, 'estimator__max_depth': 6, 'estimator__n_estimators': 320}
Balanced accuracy : 0.8818357724056404
log_loss : 0.09001655093732115
Training_acc : 0.977604788568028
Testing_acc : 0.9671694044817473
Precision : 0.9665275676000366
f1 score : 0.966329614769784 
Recall : 0.9671694044817473


In [31]:
params = {
    "estimator__learning_rate" : [0.15, 0.5, 0.9],
    "estimator__n_estimators" : [320, 350, 450],
    "estimator__max_depth" : [6, 7, 8]
    
}

xgb_grid3 = GridSearchCV(
    xgb_pipeline,
    params,
    cv=cv,
    refit = True,
    n_jobs = -1,
    scoring = "balanced_accuracy",
    verbose = 2
)

train_n_eval(xgb_grid3, "XG Boost v3")

Fitting 5 folds for each of 27 candidates, totalling 135 fits
{'estimator__learning_rate': 0.5, 'estimator__max_depth': 6, 'estimator__n_estimators': 450}
Balanced accuracy : 0.8828704190140576
log_loss : 0.10442707522539424
Training_acc : 0.9960391524945611
Testing_acc : 0.9649204159469517
Precision : 0.9641494370681039
f1 score : 0.9642225734218584 
Recall : 0.9649204159469517


In [32]:
params = {
    "estimator__learning_rate" : [0.5],
    "estimator__n_estimators" : [450, 600],
    "estimator__max_depth" : [6]
    
}

xgb_grid4 = GridSearchCV(
    xgb_pipeline,
    params,
    cv=cv,
    refit = True,
    n_jobs = -1,
    scoring = "balanced_accuracy",
    verbose = 2
)

train_n_eval(xgb_grid4, "XG Boost v4")

Fitting 5 folds for each of 2 candidates, totalling 10 fits
{'estimator__learning_rate': 0.5, 'estimator__max_depth': 6, 'estimator__n_estimators': 450}
Balanced accuracy : 0.8828704190140576
log_loss : 0.10442707522539424
Training_acc : 0.9960391524945611
Testing_acc : 0.9649204159469517
Precision : 0.9641494370681039
f1 score : 0.9642225734218584 
Recall : 0.9649204159469517


## Light GBM

In [21]:
import lightgbm as lgbm

lgbm_pipeline = Pipeline(
    [
        ("estimator", lgbm.LGBMClassifier(random_state=42))
    ]
)

params = {
    "estimator__max_depth" : [3,5],
    "estimator__learning_rate" : [0.001,0.01,0.1],
    "estimator__n_estimators" : [100,300]
}

lgbm_grid = GridSearchCV(
    lgbm_pipeline,
    params,
    cv = cv,
    scoring="balanced_accuracy",
    refit = True,
    n_jobs = -1,
    verbose = 2
)
train_n_eval(lgbm_grid, "Light GBM")

Fitting 5 folds for each of 12 candidates, totalling 60 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.147273 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1831
[LightGBM] [Info] Number of data points in the train set: 517566, number of used features: 29
[LightGBM] [Info] Start training from score -0.152365
[LightGBM] [Info] Start training from score -2.852885
[LightGBM] [Info] Start training from score -2.481146
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posi

In [22]:
params = {
    "estimator__max_depth" : [4,5,6],
    "estimator__learning_rate" : [0.09,0.1,0.15],
    "estimator__n_estimators" : [250,300,350]
}

lgbm_grid2 = GridSearchCV(
    lgbm_pipeline,
    params,
    cv = cv,
    scoring="balanced_accuracy",
    refit = True,
    n_jobs = -1,
    verbose = 2
)
train_n_eval(lgbm_grid2, "Light GBM v2")

Fitting 5 folds for each of 27 candidates, totalling 135 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.142408 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1831
[LightGBM] [Info] Number of data points in the train set: 517566, number of used features: 29
[LightGBM] [Info] Start training from score -0.152365
[LightGBM] [Info] Start training from score -2.852885
[LightGBM] [Info] Start training from score -2.481146
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with pos

In [33]:
params = {
    "estimator__max_depth" : [5],
    "estimator__learning_rate" : [0.15, 0.3],
    "estimator__n_estimators" : [350, 400]
}

lgbm_grid3 = GridSearchCV(
    lgbm_pipeline,
    params,
    cv = cv,
    scoring="balanced_accuracy",
    refit = True,
    n_jobs = -1,
    verbose = 2
)
train_n_eval(lgbm_grid3, "Light GBM v3")

Fitting 5 folds for each of 4 candidates, totalling 20 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.040714 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1831
[LightGBM] [Info] Number of data points in the train set: 517566, number of used features: 29
[LightGBM] [Info] Start training from score -0.152365
[LightGBM] [Info] Start training from score -2.852885
[LightGBM] [Info] Start training from score -2.481146
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

## Cat Boost

In [23]:
import catboost as cat

cat_pipeline = Pipeline(
    [
        ("estimator", cat.CatBoostClassifier(random_state=42))
    ]
)

params = {
    "estimator__iterations" : [100,300],
    "estimator__learning_rate" : [0.001,0.01,0.1],
    "estimator__depth" : [3,5]
}

cat_grid = GridSearchCV(
    cat_pipeline,
    params,
    cv=cv,
    scoring = "balanced_accuracy",
    refit = True,
    n_jobs = -1,
    verbose = 2
)
train_n_eval(cat_grid, "Cat Boost")

Fitting 5 folds for each of 12 candidates, totalling 60 fits
0:	learn: 0.9303786	total: 296ms	remaining: 1m 28s
1:	learn: 0.8033967	total: 459ms	remaining: 1m 8s
2:	learn: 0.7048462	total: 610ms	remaining: 1m
3:	learn: 0.6252366	total: 758ms	remaining: 56.1s
4:	learn: 0.5600204	total: 926ms	remaining: 54.6s
5:	learn: 0.5047130	total: 1.08s	remaining: 52.9s
6:	learn: 0.4582898	total: 1.24s	remaining: 51.8s
7:	learn: 0.4184567	total: 1.39s	remaining: 50.9s
8:	learn: 0.3834661	total: 1.55s	remaining: 50.3s
9:	learn: 0.3541578	total: 1.7s	remaining: 49.3s
10:	learn: 0.3277540	total: 1.89s	remaining: 49.6s
11:	learn: 0.3053466	total: 2.05s	remaining: 49.1s
12:	learn: 0.2849189	total: 2.21s	remaining: 48.7s
13:	learn: 0.2672169	total: 2.35s	remaining: 48.1s
14:	learn: 0.2516055	total: 2.49s	remaining: 47.4s
15:	learn: 0.2380861	total: 2.64s	remaining: 46.8s
16:	learn: 0.2255365	total: 2.79s	remaining: 46.5s
17:	learn: 0.2140167	total: 2.96s	remaining: 46.3s
18:	learn: 0.2040196	total: 3.1s	r

In [39]:
# for voting and Stacking
def train_n_eval2(model, model_name):
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_pred = model.predict(X_test)

    evaluation = f"Balanced accuracy : {balanced_accuracy_score(y_test, y_pred)}\nTraining_acc : {accuracy_score(y_train, y_train_pred)}\nTesting_acc : {accuracy_score(y_test, y_pred)}\nPrecision : {precision_score(y_test, y_pred, average='weighted')}\nf1 score : {f1_score(y_test, y_pred, average='weighted')} \nRecall : {recall_score(y_test, y_pred, average='weighted')}"

    results[model_name] = {
        "Balanced accuracy" : balanced_accuracy_score(y_test, y_pred),
        "log_loss" : None,
        "Accuracy" : accuracy_score(y_test, y_pred),
        "Precision" : precision_score(y_test, y_pred, average='weighted'),
        "f1 score" : f1_score(y_test, y_pred, average='weighted'),
        "Recall" : recall_score(y_test, y_pred, average='weighted'),
        "Best_param" : None
    }
    print(evaluation)

## Voting

In [25]:
from sklearn.ensemble import VotingClassifier

voting_model = VotingClassifier(
    estimators=[
        ("xg", xgb_grid.best_estimator_),
        ("lgbm", lgbm_grid2.best_estimator_),
        ("hgb", gb_grid2.best_estimator_)
    ],
    voting='soft',
    n_jobs=-1,
    verbose=2
)

train_n_eval2(voting_model, "Voting")

Balanced accuracy : 0.8775852880418186
log_loss : 0.08930902319833493
Training_acc : 0.9719881135932422
Testing_acc : 0.9674418335052921
Precision : 0.966909841583303
f1 score : 0.966468564542903 
Recall : 0.9674418335052921


C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:259: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:259: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


In [40]:

voting_model2 = VotingClassifier(
    estimators=[
        ("dt", dtc_grid.best_estimator_),
        ("xgb", xgb_grid3.best_estimator_),
        ("lgbm", lgbm_grid3.best_estimator_)
    ],
    voting='soft',
    n_jobs=-1,
    verbose=2
)

train_n_eval2(voting_model2, "Voting v2(soft)")

Balanced accuracy : 0.8830659789749217
Training_acc : 0.9893153723389867
Testing_acc : 0.9654246994586198
Precision : 0.9646694581393396
f1 score : 0.9647053214429748 
Recall : 0.9654246994586198


In [41]:

voting_model3 = VotingClassifier(
    estimators=[
        ("dt", dtc_grid.best_estimator_),
        ("xgb", xgb_grid3.best_estimator_),
        ("lgbm", lgbm_grid3.best_estimator_)
    ],
    voting='hard',
    n_jobs=-1,
    verbose=2
)

train_n_eval2(voting_model3, "Voting v2(hard)")

Balanced accuracy : 0.8818108516949533
Training_acc : 0.9889559978823956
Testing_acc : 0.9667288809543131
Precision : 0.966049344235995
f1 score : 0.9659100329997397 
Recall : 0.9667288809543131


## Stacking

In [27]:
from sklearn.ensemble import StackingClassifier

stacking_model = StackingClassifier(
    estimators = [
        ("xg", xgb_grid.best_estimator_),
        ("lgbm", lgbm_grid2.best_estimator_),
        ("hgb", gb_grid2.best_estimator_)
    ],
    final_estimator=LogisticRegression(max_iter=5000, random_state = 42),
    cv = cv,
    stack_method="predict_proba",
    passthrough = False,
    n_jobs = -1,
    verbose = 2
)

train_n_eval2(stacking_model, "Stacking")

Balanced accuracy : 0.8831168409255777
log_loss : 0.10100221061596693
Training_acc : 0.9707476920817828
Testing_acc : 0.9675403716627444
Precision : 0.9669006691402127
f1 score : 0.9667361295046414 
Recall : 0.9675403716627444


In [36]:
stacking_model2 = StackingClassifier(
    estimators = [
        ("dt", dtc_grid.best_estimator_),
        ("xgb", xgb_grid3.best_estimator_),
        ("lgbm", lgbm_grid3.best_estimator_)
    ],
    final_estimator=LogisticRegression(max_iter=5000, random_state = 42),
    cv = cv,
    stack_method="predict_proba",
    passthrough = False,
    n_jobs = -1,
    verbose = 2
)

train_n_eval2(stacking_model2, "Stacking v2")

Balanced accuracy : 0.8815396454621546
log_loss : 0.10500453450152121
Training_acc : 0.9768068227047372
Testing_acc : 0.9670476808754825
Precision : 0.9663838658225649
f1 score : 0.9662220074225986 
Recall : 0.9670476808754825


In [106]:
stacking_model3 = StackingClassifier(
    estimators = [
        ("xg", xgb_grid.best_estimator_),
        ("lgbm", lgbm_grid2.best_estimator_),
        ("hgb", gb_grid2.best_estimator_)
    ],
    final_estimator=RandomForestClassifier(n_estimators=100, random_state=42),
    cv = cv,
    stack_method="predict_proba",
    passthrough = False,
    n_jobs = -1,
    verbose = 2
)

train_n_eval2(stacking_model3, "Stacking v3")

Balanced accuracy : 0.8802254016706096
Training_acc : 0.9693275833420278
Testing_acc : 0.9661898192694265
Precision : 0.9654972319687481
f1 score : 0.9653407662811945 
Recall : 0.9661898192694265


In [108]:
stacking_model4 = StackingClassifier(
    estimators = [
        ("dt", dtc_grid.best_estimator_),
        ("lgbm", lgbm_grid3.best_estimator_),
        ("xg", xgb_grid4.best_estimator_)
    ],
    final_estimator=RandomForestClassifier(n_estimators=100, random_state=42),
    cv = cv,
    stack_method="predict_proba",
    passthrough = False,
    n_jobs = -1,
    verbose = 2
)

train_n_eval2(stacking_model4, "Stacking v4")

Balanced accuracy : 0.87709521697696
Training_acc : 0.9742699481805219
Testing_acc : 0.9652913831279489
Precision : 0.9645395209786838
f1 score : 0.964407298844362 
Recall : 0.9652913831279489


In [111]:
stacking_model5 = StackingClassifier(
    estimators = [
        ("xg", xgb_grid.best_estimator_),
        ("lgbm", lgbm_grid2.best_estimator_),
        ("hgb", gb_grid2.best_estimator_)
    ],
    final_estimator=LogisticRegression(max_iter=5000, random_state = 42),
    cv = cv,
    stack_method="predict_proba",
    passthrough = True,
    n_jobs = -1,
    verbose = 2
)

train_n_eval2(stacking_model5, "Stacking v5")

C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Balanced accuracy : 0.8826206099956616
Training_acc : 0.9716248748951825
Testing_acc : 0.9672041826549658
Precision : 0.9665493949567563
f1 score : 0.9663872592907492 
Recall : 0.9672041826549658


# Training Results

In [112]:
df_results = pd.DataFrame(results).T
df_results.columns = ["Balanced Accuracy","log_loss", "Accuracy", "Precision", "f1 score", "Recall", "Best Param"]
df_results = df_results.sort_values(by="Balanced Accuracy", ascending = False)
df_results.to_csv("results.csv", index=True)

# Testing 

In [113]:
tdata = pd.read_csv("test.csv")

In [114]:
test_id = tdata["id"]

In [115]:
tdata = tdata.drop(columns=['id'])

In [116]:
# Handling missing values

#------missing indicators-------------
tdata['missing_sleep_duration'] = tdata['sleep_duration'].isnull().astype(int)
tdata['missing_heart_rate'] = tdata['heart_rate'].isnull().astype(int)
tdata['missing_bmi'] = tdata['bmi'].isnull().astype(int)
tdata['missing_calorie_expenditure'] = tdata['calorie_expenditure'].isnull().astype(int)
tdata['missing_step_count'] = tdata['step_count'].isnull().astype(int)
tdata['missing_exercise_duration'] = tdata['exercise_duration'].isnull().astype(int)
tdata['missing_water_intake'] = tdata['water_intake'].isnull().astype(int)
tdata['missing_diet_type'] = tdata['diet_type'].isnull().astype(int)
tdata['missing_stress_level'] = tdata['stress_level'].isnull().astype(int)
tdata['missing_sleep_quality'] = tdata['sleep_quality'].isnull().astype(int)
tdata['missing_physical_activity_level'] = tdata['physical_activity_level'].isnull().astype(int)
tdata['missing_smoking_alcohol'] = tdata['smoking_alcohol'].isnull().astype(int)
tdata['missing_gender'] = tdata['gender'].isnull().astype(int)

#------ fill missing values-----------
tdata['sleep_duration'] = tdata['sleep_duration'].fillna(tdata['sleep_duration'].median())
tdata['heart_rate'] = tdata['heart_rate'].fillna(tdata['heart_rate'].median())
tdata['bmi'] = tdata['bmi'].fillna(tdata['bmi'].median())
tdata['calorie_expenditure'] = tdata['calorie_expenditure'].fillna(tdata['calorie_expenditure'].median())
tdata['step_count'] = tdata['step_count'].fillna(tdata['step_count'].median())
tdata['exercise_duration'] = tdata['exercise_duration'].fillna(tdata['exercise_duration'].median())
tdata['water_intake'] = tdata['water_intake'].fillna(tdata['water_intake'].median())
tdata['diet_type'] = tdata['diet_type'].fillna(tdata['diet_type'].mode()[0])
tdata['stress_level'] = tdata['stress_level'].fillna(tdata['stress_level'].mode()[0])
tdata['sleep_quality'] = tdata['sleep_quality'].fillna(tdata['sleep_quality'].mode()[0])
tdata['physical_activity_level'] = tdata['physical_activity_level'].fillna(tdata['physical_activity_level'].mode()[0])
tdata['smoking_alcohol'] = tdata['smoking_alcohol'].fillna(tdata['smoking_alcohol'].mode()[0])
tdata['gender'] = tdata['gender'].fillna(tdata['gender'].mode()[0])


In [117]:
# Encoding
#-----manual mapping--------
stress_mapping = {
    'low' : 0,
    'medium' : 1,
    'high' : 2
}
tdata['stress_level'] = tdata['stress_level'].map(stress_mapping)

sleep_mapping = {
    'poor' : 0,
    'average' : 1,
    'good' : 2
}
tdata['sleep_quality'] = tdata['sleep_quality'].map(sleep_mapping)

physical_mapping = {
    'sedentary' : 0,
    'moderate' : 1,
    'active' : 2
}
tdata['physical_activity_level'] = tdata['physical_activity_level'].map(physical_mapping)

#-----OHE------------

encoded = ohe.transform(tdata[['diet_type', 'smoking_alcohol', 'gender']])
encoded_data = pd.DataFrame(encoded, columns = ohe.get_feature_names_out(), index = tdata.index)
tdata = pd.concat([tdata.drop(columns = ['diet_type', 'smoking_alcohol', 'gender']), encoded_data], axis = 1)

In [118]:
y_pred = stacking_model5.predict(tdata)

In [119]:
y_pred = le.inverse_transform(y_pred)

In [120]:
submission = pd.DataFrame({
    "id": test_id,
    "health_condition": y_pred
})

In [121]:
submission.to_csv("Stacking_V_5.csv", index=False)